## Datascout Tags - Helper functions

# Common Notebooks / Scripts

- `one_password`: this will give access to the 1Password methods that load credentials (from client ID) for the Datascout environment.
- `imis_client`: this will give access to the iMIS API based on the client credentials

In [ ]:
import sys
import json
from pathlib import Path
from onepassword import ItemField, ItemFieldType
import datetime
import time

import numpy as np
import unicodedata
import re
import os
import requests

import pandas as pd

# Add the root of the repo to sys.path
repo_root = Path.cwd().resolve().parents[0]  # keep 0 if inside notebook directory (not nested)
sys.path.append(str(repo_root))


# Load the get-secrets notebook as a module
from common.one_password import get_client_secrets

# Load the IMIS client module
from common.imis_client import IMISClient

# Client Config Setup

`Client ID` - Used to fetch all client credentials and secrets from 1password

In [ ]:
IMIS_BASE_URL = "WILL BE PROVIDED BY CALLING NOTEBOOK"
imis_client = None

In [ ]:
# # --- SET CLIENT ID HERE ---
# CLIENT_ID = "demo42"  # Replace with client ID

# # Get the client secrets from the notebook
# client_secrets = await get_client_secrets(CLIENT_ID)

# if not client_secrets:
#     print("No client secrets found. Please check the CLIENT_ID or the one-password script.")
#     raise SystemExit(1)

# IMIS_BASE_URL = client_secrets["imis_base_url"]
# IMIS_USERNAME = client_secrets["username"]
# IMIS_PWD = client_secrets["password"]

# print(f"IMIS_BASE_URL: {IMIS_BASE_URL}")
# print(f"IMIS_USERNAME: {IMIS_USERNAME}")

# imis_client = IMISClient(IMIS_BASE_URL, IMIS_USERNAME, IMIS_PWD)

## Helper functions

In [ ]:
def merge_id_dataframes(dfs_array):
    if not isinstance(dfs_array, list):
        raise ValueError("Input must be a list of DataFrames.")
    
    id_series_list = []

    for i, df in enumerate(dfs_array):
        if df is None or df.empty:
            continue
        if "id" not in df.columns:
            raise ValueError(f"DataFrame at index {i} is missing 'id' column.")
        id_series_list.append(df["id"])

    if not id_series_list:
        return pd.DataFrame(columns=["id"])

    merged_ids = pd.concat(id_series_list, ignore_index=True).drop_duplicates()
    return pd.DataFrame({"id": merged_ids})
# Function to remove accents from a string
def remove_accents(input_str):
    if input_str is None or pd.isna(input_str):
        return None
    nkfd_form = unicodedata.normalize('NFKD', str(input_str))
    return u"".join([c for c in nkfd_form if not unicodedata.combining(c)])

# Helper function to format tags (replacing the previously implied format_tag function)
def format_tag(value):
    if value is None or pd.isna(value):
        return ""
    # Remove accents and lowercase
    normalized = remove_accents(str(value).lower())
    # Remove apostrophes
    no_apostrophes = normalized.replace("'", "")
    # Replace spaces and underscores with dashes
    dash_spaces = re.sub(r"[ _]+", "-", no_apostrophes)
    # Remove all non-alphanumeric and non-dash chars
    alphanum_dash = re.sub(r"[^a-z0-9\-]", "", dash_spaces)
    # Collapse multiple dashes
    compressed = re.sub(r"-+", "-", alphanum_dash)
    # Trim leading/trailing dashes
    treated_tag = re.sub(r"(^-+)|(-+$)", "", compressed)
    return treated_tag if treated_tag != "blank" else ""
def diff_tags(current_objs, new_tags):
    """
    current_objs: list of dicts, each with at least "tag_value" (string) and "ordinal" (int).
    new_tags:     list of strings (e.g. ["donor", "age-45-60", ...]).

    Returns: list of actions where each action is either:
      {"operation": "delete", "tag": <tag_value>, "ordinal": <ordinal>}
    or
      {"operation": "add",    "tag": <tag_value>}
    
    • If current_objs contains duplicates of a tag not in new_tags, or more copies than new_tags, 
      emit a "delete" for each extra copy (include ordinal).
    • If new_tags has a tag not in current_objs, emit a single "add".
    • Comparison is case-insensitive and ignores leading/trailing whitespace.
    """
    # 1) Normalize new_tags into a set of lowercase keys, and keep a map to original casing
    normalized_new = {t.strip().lower() for t in new_tags}
    new_map = {t.strip().lower(): t for t in new_tags}

    # 2) Group current_objs by normalized tag_value
    from collections import defaultdict
    current_group = defaultdict(list)
    for obj in current_objs:
        norm = obj["tag_value"].strip().lower()
        current_group[norm].append(obj)

    actions = []

    # 3) Determine deletions: for each normalized tag present in current_group,
    #    compare how many copies exist vs. new_tags (new count is 1 if in normalized_new, else 0).
    for norm_tag, obj_list in current_group.items():
        curr_count = len(obj_list)
        new_count = 1 if norm_tag in normalized_new else 0
        if curr_count > new_count:
            # Delete (curr_count - new_count) objects. 
            # We'll take the first ones in obj_list for simplicity.
            to_delete_count = curr_count - new_count
            for obj in obj_list[:to_delete_count]:
                actions.append({
                    "operation": "delete",
                    "tag": obj["tag_value"],
                    "ordinal": obj["ordinal"]
                })

    # 4) Determine additions: any tag in new_tags not present in current_group
    for norm_tag in normalized_new:
        if norm_tag not in current_group:
            # Add exactly one copy of that tag
            actions.append({
                "operation": "add",
                "tag": new_map[norm_tag]
            })

    return actions
def add_tag_record(id, tag_type, tag_value):

    # we need to use the iMIS soap data structure :( as a template 
    # note the variables in {{imis_id}}, {{tag_type}} and {{tag_value}}

    tag_template = """{
        "$type": "Asi.Soa.Core.DataContracts.GenericEntityData, Asi.Contracts",
        "EntityTypeName": "Datascout_Tags",
        "PrimaryParentEntityTypeName": "Party",
        "Identity": {
            "$type": "Asi.Soa.Core.DataContracts.IdentityData, Asi.Contracts",
            "EntityTypeName": "Datascout_Tags",
            "IdentityElements": {
                "$type": "System.Collections.ObjectModel.Collection`1[[System.String, mscorlib]], mscorlib",
                "$values": [
                    "{{imis_id}}"
                ]
            }
        },
        "PrimaryParentIdentity": {
            "$type": "Asi.Soa.Core.DataContracts.IdentityData, Asi.Contracts",
            "EntityTypeName": "Party",
            "IdentityElements": {
                "$type": "System.Collections.ObjectModel.Collection`1[[System.String, mscorlib]], mscorlib",
                "$values": [
                    "{{imis_id}}"
                ]
            }
        },
        "Properties": {
            "$type": "Asi.Soa.Core.DataContracts.GenericPropertyDataCollection, Asi.Contracts",
            "$values": [
                {
                    "$type": "Asi.Soa.Core.DataContracts.GenericPropertyData, Asi.Contracts",
                    "Name": "ID",
                    "Value": "{{imis_id}}"
                },
                {
                    "$type": "Asi.Soa.Core.DataContracts.GenericPropertyData, Asi.Contracts",
                    "Name": "TagType",
                    "Value": "{{tag_type}}"
                },
                {
                    "$type": "Asi.Soa.Core.DataContracts.GenericPropertyData, Asi.Contracts",
                    "Name": "TagValue",
                    "Value": "{{tag_value}}"
                }
            ]
        }
    }
    """

    tag_template = tag_template.replace("{{imis_id}}", str(id))
    tag_template = tag_template.replace("{{tag_type}}", tag_type)
    tag_template = tag_template.replace("{{tag_value}}", tag_value)
    # print(tag_template)
    
    url = f"{IMIS_BASE_URL}/api/Datascout_Tags?ID={id}"
    return imis_client.make_request("POST", url, tag_template, None)

def process_system_tags(id, tags_to_process, current_system_tags):
    for tag_action in tags_to_process:
        tag_value = tag_action['tag']
        operation = tag_action['operation']

        if operation == 'delete':
            start_time = time.time()
            ordinal = tag_action['ordinal']
            url = f"{IMIS_BASE_URL}/api/Datascout_Tags/~{id}|{ordinal}"
            imis_client.make_request("DELETE", url, None, None)
            elapsed_time = time.time() - start_time
            print(f"\tDeleted tag '{tag_value}' ({id}|{ordinal}) in {elapsed_time:.2f}s")
        
        elif operation == 'add':
            start_time = time.time()
            add_tag_record(id, "System", tag_value)
            elapsed_time = time.time() - start_time
            print(f"\tAdded tag '{tag_value}' to {id} in {elapsed_time:.2f}s")
              
        else:
            print(f"\t⚠️ Unknown operation '{operation}' for tag '{tag_value}'")

def get_df_from_iqa(iqa_path, params = None):
    imis_data  = imis_client.fetch_iqa(iqa_path, None, 500, params)
    df = pd.DataFrame(imis_data)
    return df

In [ ]:
from openai import OpenAI
client = OpenAI()

def generate_bins_from_series_stats(series_stats):
    """
    Generates recommended bins and labels based on stats of a numeric pandas Series.
    Parameters:
    series_stats (pd.Series): A Series with min, max, mean of the target variable.
    Returns:
    str: Suggested bins, labels, and exaplanation.
    """
    min_val = series_stats['min']
    max_val = series_stats['max']
    mean_val = series_stats['mean']
    
    prompt = f"""
    Create meaningful bins for data: Min={min_val}, Max={max_val}, Mean={mean_val:.2f}

    Start first bin at {min_val-0.5} to include minimum value.
    Use "X-to-Y" format for all labels except the last one which should be "X+".

    Example:
    Line 1: bins = [{min_val-0.5}, breakpoint1+0.5, breakpoint2+0.5, breakpoint3+0.5, float('inf')]
    Line 2: labels = ['5-to-7', '8-to-11', '12+']

    Output format (strictly):
    bins = []
    labels = []
    A few bullet points explaining the rationale behind the binning. Keep it simple, tied to the stats, and avoid any numbering or formatting marks.
    
    Additional rules:
    - All breakpoints must end in .5 to create clean integer ranges
    - Match the number of labels to the number of bin ranges
    - ALWAYS end with float('inf'), never with the actual maximum value
    - Keep binning interpretable and relevant to the mean and max
    - Do not number the lines, do not wrap in quotes, do not use markdown or code blocks. Just return raw clean output in the format above.
    """
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()